# **Flood Event Analysis for Nairobi**

**Author:** Stacy Mwangi  
**Date:** 2025–2026  
**GitHub Repository:** [CRUM_Nairobi](https://github.com/stacyfuende/CRUM_Nairobi)

---

## Research Question

> *Where does recurring flooding occur in Nairobi, and which specific roads are most consistently affected?*

## Project Overview

This notebook analyses **10 years of Sentinel-1 SAR flood detections** (2014–2024) across Nairobi to identify:

- **Where** flooding recurs most persistently across sub-counties
- **Which roads** fall inside chronic flood zones and face repeated disruption
- **How many people** live in areas at risk of recurrent flooding

The analysis combines satellite radar data with population density and OpenStreetMap road networks to produce an actionable flood risk map for urban planners, emergency responders, and researchers.

### Why SAR?

Synthetic Aperture Radar \(SAR\) satellites like Sentinel\-1 penetrate cloud cover and work at night which is essential for tropical cities like Nairobi where flood events often coincide with heavy cloud. The AI for Good Lab dataset processes 10 years of these observations into point\-level flood detections at **20\-metre resolution**.


## 1. Loading Data

### Data Sources

| Dataset | Source | Purpose |
| :------------------- | :-------------------------------------------------------------------------------------------------------------------------- | :--------------------------------------------------- |
| **Flood Detections** | [AI for Good Lab \- Hugging Face](https://huggingface.co/datasets/ai-for-good-lab/ai4g-flood-dataset/tree/main/S03/S03E036) | Core flood detection points, 2014–2024 |
| **Population** | [WorldPop Population Counts](https://hub.worldpop.org/geodata/summary?id=73999) | 100m\-resolution population density for Kenya |
| **Roads** | OpenStreetMap via `osmdata` \(R\) | Road network for identifying at\-risk infrastructure |

**Paper:** _Mapping global floods with 10 years of satellite radar data_ \- Nature Communications \(2025\)  
[https://doi.org/10.1038/s41467\-025\-60973\-1](https://doi.org/10.1038/s41467-025-60973-1)

### Tile Selection

The AI4G dataset is organised in **3° × 3° tiles** (~330 km × 330 km). Nairobi sits at approximately 1.28°S, 36.82°E.  
To find the correct tile: round *down* to the nearest 3°.  

- Latitude: −1.28° → **S03** (i.e. −3° to 0°)  
- Longitude: 36.82° → **E036** (i.e. 36° to 39°)  
- **Tile: `S03E036`** → file: `S03E036-post-processing.parquet`

### Parquet Schema (Key Columns)

| Column | Description |
|---|---|
| `lat`, `lon` | Flood detection coordinates (WGS84) |
| `year`, `month`, `day` | Date of detection |
| `dem_metric_2` | Max terrain slope within 240m — high values indicate false positives on steep terrain |
| `soil_moisture_zscore` | Anomaly vs. historical baseline — positive = unusually wet |
| `soil_moisture_sca` | Soil moisture from SCA algorithm (%) |
| `soil_moisture` | LPRM soil moisture (%) |
| `temp` | Average daily minimum temperature (°C) |
| `land_cover` | ESA WorldCover class (60 = bare ground, often excluded) |
| `edge_false_positives` | Flag (1) for detections near tile edges — should be excluded |

> **Note on raw data paths:** Update `PARQUET_PATH` and `WORLDPOP_PATH` to match your local directory structure before running this notebook.



In [4]:
# Load libraries
library(tidyverse)
library(sf)
library(terra)
library(osmdata)
library(ggplot2)
library(MASS)       
library(scales)
library(patchwork)
library(ggspatial)
library(ggrepel)
library(cowplot)
library(arrow)      
library(ggnewscale)
library(duckdb)
library(purrr)
library(tidyr)

setwd("/home/user/Capstone-Project/CRUM_Nairobi/data")

con <- dbConnect(duckdb(), dbdir = ":memory:")

# Define Nairobi bounding box (city extent)
nairobi_bbox <- c(
  xmin = 36.65, ymin = -1.45,
  xmax = 37.10, ymax = -1.10
)

# Colour palette 
COL_BG        <- "#F5EFE0"       # parchment
COL_FLOOD_LOW <- "#A8C5C0"       # pale teal
COL_FLOOD_MID <- "#3A7D8C"       # mid teal
COL_FLOOD_HI  <- "#0D3B52"       # deep navy (chronic)
COL_POP       <- "#D4622A"       # burnt orange for population
COL_ROAD      <- "#BFB9A8"       # muted warm grey roads
COL_ROAD_RISK <- "#E8C84A"       # amber - roads inside flood zone
COL_TEXT      <- "#2C2416"       # dark warm brown
COL_ACCENT    <- "#C0392B"       # red accent for callout


PARQUET_PATH    <- "raw_data/S03E036-post-processing.parquet"
message("Flood parquet loaded")
WORLDPOP_PATH   <- "raw_data/ken_pop_2026_CN_100m_R2025A_v1.tif"  # download from worldpop.org
message("Population data loaded")

# 1. Read ONLY Nairobi-relevant rows from the parquet

flood_nairobi <- dbGetQuery(con, glue::glue("
    SELECT
        lon, lat, year, month,
        dem_metric_2, soil_moisture_sca, soil_moisture_zscore,
        soil_moisture, temp, land_cover, edge_false_positives
    FROM read_parquet('{PARQUET_PATH}')
    WHERE lon BETWEEN {nairobi_bbox['xmin']} AND {nairobi_bbox['xmax']}
      AND lat BETWEEN {nairobi_bbox['ymin']} AND {nairobi_bbox['ymax']}
"))

summary(flood_nairobi)

# 2. Apply your domain filters

flood_filtered <- flood_nairobi %>%
  filter(
    dem_metric_2       < 10,
    soil_moisture_sca  > 1,
    soil_moisture_zscore > -2,
    soil_moisture      > 20,
    temp               > 0,
#    land_cover         != 60,
    edge_false_positives == 0
  )

# Load supplementary datasets
subcounties <- st_read("raw_data/gadm41_KEN_2.shp")%>% filter(NAME_1 == "Nairobi")

Flood parquet loaded



Population data loaded



      lon             lat              year          month       
 Min.   :36.65   Min.   :-1.450   Min.   :2014   Min.   : 1.000  
 1st Qu.:36.86   1st Qu.:-1.432   1st Qu.:2017   1st Qu.: 1.000  
 Median :36.93   Median :-1.403   Median :2017   Median : 7.000  
 Mean   :36.94   Mean   :-1.379   Mean   :2017   Mean   : 5.458  
 3rd Qu.:37.04   3rd Qu.:-1.348   3rd Qu.:2018   3rd Qu.: 9.000  
 Max.   :37.10   Max.   :-1.100   Max.   :2024   Max.   :12.000  
  dem_metric_2   soil_moisture_sca soil_moisture_zscore soil_moisture   
 Min.   : 0.00   Min.   : 0.800    Min.   :-1.9400      Min.   : 0.200  
 1st Qu.: 2.08   1st Qu.: 1.000    1st Qu.:-0.9000      1st Qu.: 2.000  
 Median : 3.26   Median : 1.300    Median :-0.7200      Median : 4.300  
 Mean   : 3.73   Mean   : 2.547    Mean   :-0.4656      Mean   : 6.729  
 3rd Qu.: 4.73   3rd Qu.: 3.500    3rd Qu.:-0.3500      3rd Qu.: 8.300  
 Max.   :47.75   Max.   :28.100    Max.   : 5.1800      Max.   :64.000  
      temp         land_cov

Reading layer `gadm41_KEN_2' from data source 
  `/home/user/Capstone-Project/CRUM_Nairobi/data/raw_data/gadm41_KEN_2.shp' 
  using driver `ESRI Shapefile'
Simple feature collection with 300 features and 13 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: 33.90959 ymin: -4.720417 xmax: 41.92622 ymax: 5.061166
Geodetic CRS:  WGS 84


## 2. Data Cleaning & Filtering

### Why Filter?

The raw parquet contains all SAR flood detections, including **false positives** from:

- **Steep terrain** \- radar backscatter mimics water on rough slopes
- **Dry desert soils** \- very low soil moisture can produce spurious detections
- **Tile edges** \- geometric artefacts near 3° boundaries
- **Freezing conditions** \- ice/snow is misclassified as water
- **Bare ground** \- land cover class 60 produces systematic false positives

### Recommended Filters (for multi-month aggregation)

```r
dem_metric_2       < 15    # Exclude steep terrain (dataset default: < 10; relaxed here for urban hills)
soil_moisture_sca  > 0     # Some SCA moisture signal required
soil_moisture_zscore > -2  # Not abnormally dry
soil_moisture      > 0     # Basic moisture threshold
temp               > 0     # Above freezing
edge_false_positives == 0  # Remove edge artefacts
```

> **Tip:** For analysing a _single flood event_, you can skip these filters \- they are intended for long\-term aggregation to reduce false positive accumulation. The thresholds used here are slightly relaxed from the paper defaults to retain detections in Nairobi's more variable terrain.

### Grid Aggregation

Rather than working with individual 20m points, detections are **binned into a 100m grid** in UTM projection \(EPSG:32737 \- UTM Zone 37S\). This:

- Speeds up computation
- Removes near-duplicate detections from overlapping satellite passes
- Allows counting **distinct flood months per cell** as a recurrence measure

### Kernel Density Estimation (KDE)

Grid centroids are weighted by `n_months` and passed to a 2D KDE (`MASS::kde2d`) to produce smooth **flood hotspot contours** for the final map visualisation.



In [5]:

# --- Parameters ---
crs_metric <- 32737      # UTM 37S
cell_m     <- 100
bb_deg     <- nairobi_bbox

# 1) Points in WGS84
flood_pts_wgs <- st_as_sf(
  flood_filtered %>% dplyr::select(lon, lat, year, month),
  coords = c("lon", "lat"), crs = 4326
)

# 2) Project to meters
flood_pts_m <- st_transform(flood_pts_wgs, crs_metric)

# 3) Snap to the 100 m grid
XY <- st_coordinates(flood_pts_m)
flood_binned <- flood_pts_m %>%
  dplyr::mutate(
    cell_x   = floor(XY[,1] / cell_m) * cell_m,
    cell_y   = floor(XY[,2] / cell_m) * cell_m,
    month_id = year*100L + month
  ) %>%
  st_drop_geometry()

# 4) Recurrence per grid cell
flood_recurrence_grid <- flood_binned %>%
  dplyr::group_by(cell_x, cell_y) %>%
  dplyr::summarise(
    n_months   = dplyr::n_distinct(month_id),
    n_events   = dplyr::n(),
    first_year = min(year),
    last_year  = max(year),
    .groups = "drop"
  ) %>%
  dplyr::mutate(
    risk_class = dplyr::case_when(
      n_months >= 6 ~ "Chronic (6+ months)",
      n_months >= 3 ~ "High (3–5 months)",
      n_months >= 1 ~ "Occasional (1–2 months)"
    ),
    risk_class = factor(
      risk_class,
      levels = c("Occasional (1–2 months)", "High (3–5 months)", "Chronic (6+ months)")
    )
  )

print(flood_recurrence_grid %>% dplyr::count(risk_class, sort = TRUE))

# 5) Grid polygons (UTM)
grid_polygons <- flood_recurrence_grid %>%
  dplyr::mutate(
    geometry = purrr::map2(cell_x, cell_y, ~{
      st_polygon(list(matrix(c(
        .x,            .y,
        .x + cell_m,   .y,
        .x + cell_m,   .y + cell_m,
        .x,            .y + cell_m,
        .x,            .y
      ), ncol = 2, byrow = TRUE)))
    })
  ) %>%
  st_as_sf(crs = crs_metric)
head(grid_polygons)
# 6) Centroids for KDE (WGS84)
grid_centroids <- st_centroid(grid_polygons) %>% st_transform(4326)

# ---- Build KDE input ONCE (as a plain data frame) ----
# Get lon/lat from geometry:
coords_df <- st_coordinates(grid_centroids) %>%
  as.data.frame() %>%
  dplyr::rename(lon = X, lat = Y)

# Join n_months, then expand rows by n_months weight:
kde_input_df <- grid_centroids %>%
  st_drop_geometry() %>%
  dplyr::select(n_months) %>%
  dplyr::bind_cols(coords_df) %>%
  tidyr::uncount(n_months)   # replicate rows by n_months

# --- KDE ---
message("Computing flood density contours...")

kde_fit <- MASS::kde2d(
  x    = kde_input_df$lon,
  y    = kde_input_df$lat,
  n    = 400,
  lims = c(nairobi_bbox["xmin"], nairobi_bbox["xmax"],
           nairobi_bbox["ymin"], nairobi_bbox["ymax"])
)

kde_df <- expand.grid(lon = kde_fit$x, lat = kde_fit$y) %>%
  dplyr::mutate(density = as.vector(kde_fit$z))
head(kde_df)

nrow(flood_nairobi)
nrow(flood_filtered)

# A tibble: 2 × 2
  risk_class                  n
  <fct>                   <int>
1 Occasional (1–2 months)  2205
2 High (3–5 months)           2


Registered S3 method overwritten by 'geojsonsf':
  method        from   
  print.geojson geojson



cell_x,cell_y,n_months,n_events,first_year,last_year,risk_class,geometry
<dbl>,<dbl>,<int>,<int>,<int>,<int>,<fct>,<POLYGON [m]>
239800,9867600,1,2,2018,2018,Occasional (1–2 months),"POLYGON ((239800 9867600, 2..."
239800,9867700,1,4,2018,2018,Occasional (1–2 months),"POLYGON ((239800 9867700, 2..."
240400,9852600,1,3,2021,2021,Occasional (1–2 months),"POLYGON ((240400 9852600, 2..."
242900,9850000,1,1,2018,2018,Occasional (1–2 months),"POLYGON ((242900 9850000, 2..."
243000,9840300,1,6,2018,2018,Occasional (1–2 months),"POLYGON ((243000 9840300, 2..."
243100,9840300,1,5,2018,2018,Occasional (1–2 months),"POLYGON ((243100 9840300, 2..."


Warning message:
“st_centroid assumes attributes are constant over geometries”


Computing flood density contours...



,lon,lat,density
,<dbl>,<dbl>,<dbl>
1,36.65000,-1.45,4.093627e-05
2,36.65113,-1.45,7.654595e-05
3,36.65226,-1.45,1.407489e-04
4,36.65338,-1.45,2.544938e-04
5,36.65451,-1.45,4.525016e-04
6,36.65564,-1.45,7.911785e-04


[1] 275093

[1] 4941

## 3. Exploratory Data Analysis (EDA)

### Loading the Road Network

OSM roads are fetched for Nairobi's bounding box and classified into:

- **Major** \- primary and secondary roads \(main arteries\)
- **Local** \- tertiary, residential, unclassified, living streets
- **Footpath** \- pedestrian paths \(although data for this is limited in Nairobi context\)

This classification matters for the final map: major flood-affected roads have more severe disruption impact on commuters and emergency access.

### Identifying Roads at Flood Risk

Roads are considered **at risk** if they intersect a 24**0\-metre buffer** around flood cells with ≥ 3 months of recurrence. The 240m buffer aligns with the dataset's recommended spatial buffer for raster output, accounting for the 20m detection resolution and sensor uncertainty.\(To bypass this step you can just use the existing buffer files provided in the repository\)

### Population Exposure

WorldPop 100m raster data is cropped to Nairobi's extent and spatially joined with:

1. Sub-county boundaries (GADM Level 2) → total population per sub-county
2. Chronic flood union polygon → population *inside* flood zones

This produces `pct_at_risk`: the share of each sub-county's residents living inside recurrent flood areas.



In [6]:
# User option
use_cached <- TRUE   # set to FALSE to fetch fresh OSM data

# Paths
cache_file <- "raw_data/nairobi_roads.gpkg"

message("Preparing OSM query...")

osm_query <- opq(bbox = c(
  nairobi_bbox["xmin"], nairobi_bbox["ymin"],
  nairobi_bbox["xmax"], nairobi_bbox["ymax"]
)) %>%
  add_osm_feature(
    key = "highway",
    value = c(
      "primary", "secondary", "tertiary", "residential",
      "unclassified", "living_street", "footway", "path"
    )
  )


# ===== OPTION A: LOAD FROM CACHE =====
if (use_cached && file.exists(cache_file)) {

  message("Loading OSM roads from cached file...")
  roads_raw <- read_sf(cache_file)

} else {

  # ===== OPTION B: DOWNLOAD FROM OVERPASS =====
  message("Fetching OSM roads from Overpass API...")

  osm_sf <- osmdata_sf(osm_query)

  # Check available line geometries
  if (is.null(osm_sf$osm_lines) || nrow(osm_sf$osm_lines) == 0) {
    stop("No line geometries returned from OSM.")
  }

  roads_raw <- osm_sf$osm_lines

  # Save cache for reproducibility
  dir.create("raw_data", showWarnings = FALSE)
  st_write(roads_raw, cache_file, delete_dsn = TRUE)
  message("Downloaded OSM roads saved to cache.")
}


# ===== CLEAN + STANDARDIZE =====
roads_lines <- roads_raw %>%
  dplyr::select(osm_id, name, highway, geom) %>%
  st_transform(4326)


# ===== CLASSIFY ROAD TYPES =====
roads_lines <- roads_lines %>%
  mutate(road_class = case_when(
    highway %in% c("primary", "secondary") ~ "Major",
    highway %in% c(
      "tertiary", "residential", "unclassified", "living_street"
    ) ~ "Local",
    highway %in% c("footway", "path") ~ "Footpath",
    TRUE ~ "Local"
  ))
message("Roads loaded, cleaned, and classified.")

Preparing OSM query...



Loading OSM roads from cached file...



Roads loaded, cleaned, and classified.



In [7]:
# =====================================================
# 4B. IDENTIFY ROADS AT FLOOD RISK (FINAL, CORRECTED)
# =====================================================

# 1) Flood mask in UTM (grid_polygons is already UTM: 32737)
chronic_min_months <- 2   # keep consistent across notebook

centroids_m <- sf::st_centroid(grid_polygons)  # warning is benign

flood_buffer <- centroids_m %>%
  dplyr::filter(n_months >= chronic_min_months) %>%
  sf::st_buffer(240) %>%
  sf::st_make_valid()

flood_union <- flood_buffer %>%
  sf::st_union() %>%
  sf::st_make_valid()

# 2) Roads in UTM
roads_lines_m <- roads_lines %>%
  sf::st_make_valid() %>%
  sf::st_transform(32737)

# 3) Ensure subcounties are UTM, then SPLIT ROADS BY SUBCOUNTY
subcounties_m <- sf::st_transform(subcounties, 32737)

roads_by_subcounty <- sf::st_intersection(
  roads_lines_m,
  subcounties_m %>% dplyr::select(NAME_2)   # <<-- EXPLICIT dplyr::select
) %>%
  dplyr::filter(!sf::st_is_empty(sf::st_geometry(.))) %>%
  dplyr::mutate(
    road_len_km = as.numeric(sf::st_length(sf::st_geometry(.))) / 1000
  )

# 4) Clip these road pieces to the flood union (get only flooded segments)
roads_flood_segments <- suppressWarnings(
  sf::st_intersection(roads_by_subcounty, flood_union)
) %>%
  dplyr::filter(!sf::st_is_empty(sf::st_geometry(.))) %>%
  dplyr::mutate(
    flood_len_km = as.numeric(sf::st_length(sf::st_geometry(.))) / 1000
  )

# 5) True/False flag for mapping (whole road geometries)
roads_at_risk <- roads_lines_m %>%
  dplyr::mutate(in_flood_zone = lengths(sf::st_intersects(., flood_union)) > 0)

# 6) Summarise flooded road length per subcounty
roads_subcounty <- roads_flood_segments %>%
  sf::st_drop_geometry() %>%
  dplyr::group_by(NAME_2) %>%
  dplyr::summarise(
    roads_risk_km = round(sum(flood_len_km, na.rm = TRUE), 2),
    .groups = "drop"
  )

# 7) Merge back into your subcounty stats
subcounty_stats <- subcounty_stats %>%
  dplyr::select(-dplyr::any_of("roads_risk_km")) %>%
  dplyr::left_join(roads_subcounty, by = "NAME_2") %>%
  dplyr::mutate(roads_risk_km = tidyr::replace_na(roads_risk_km, 0))

message("Subcounty stats (with road flood risk) computed successfully!")
print(subcounty_stats)


Warning message:
“st_centroid assumes attributes are constant over geometries”


Warning message:
“attribute variables are assumed to be spatially constant throughout all geometries”


ERROR: Error: object 'subcounty_stats' not found


In [0]:
# 6. NEIGHBOURHOOD LABELS (Nairobi key areas)

# --- 6. NEIGHBOURHOOD LABELS (for annotation) ---
nairobi_labels <- tibble(
  name = c("Mathare","Kibera","Mukuru","Kawangware",
           "Korogocho","Ngong River Corridor","Nairobi CBD"),
  lon  = c(36.857, 36.779, 36.873, 36.738,
           36.882, 36.800, 36.820),
  lat  = c(-1.256, -1.312, -1.305, -1.283,
           -1.246, -1.335, -1.284)
)

labels_sf <- st_as_sf(nairobi_labels, coords = c("lon","lat"), crs = 4326)
labels_sf <- st_transform(labels_sf, 32737)


# 7. COMPUTE SUMMARY STATS FOR CALLOUTS

chronic_min_months <- 2      # threshold (tune as needed)

chronic_cells <- grid_polygons %>%
  filter(n_months >= chronic_min_months) %>%
  st_make_valid()

if (nrow(chronic_cells) == 0) {
  warning("No chronic/recurrent cells found. Lower the threshold or adjust filters.")
  chronic_union <- st_sfc(st_polygon(), crs = st_crs(grid_polygons))
} else {
  chronic_union <- chronic_cells %>%
    st_union() %>%
    st_make_valid()
}
pop_sf <- st_as_sf(pop_df, coords = c("lon","lat"), crs = 4326)
pop_sf <- st_transform(pop_sf, 32737)  # match chronic_union CRS

pop_in_chronic <- pop_sf %>%
  filter(lengths(st_intersects(geometry, chronic_union)) > 0) %>%
  pull(population) %>%
  sum(na.rm = TRUE)
# roads_flood_segments is the clipped, valid geometry in UTM
roads_risk_km <- roads_flood_segments %>%
  st_length() %>%
  sum(na.rm = TRUE) %>%
  units::set_units("km") %>%
  as.numeric() %>%
  round(1)
total_events <- nrow(flood_filtered)
message(glue::glue(
  "People in chronic zones: ~{scales::comma(round(pop_in_chronic, -2))}\n",
  "Roads at risk: {roads_risk_km} km\n",
  "Total flood detections: {scales::comma(total_events)}"
))

# 8a. CHART — People at risk per subcounty
# (Requires subcounty_stats from cell 6)

chart_people <- ggplot(
  subcounty_stats %>% arrange(desc(pop_at_risk)) %>% slice_head(n = 10),
  aes(
    x    = reorder(NAME_2, pop_at_risk),
    y    = pop_at_risk,
    fill = pop_at_risk
  )
) +
  geom_col(width = 0.7) +
  geom_text(
    aes(label = scales::comma(round(pop_at_risk, -2))),
    hjust = -0.1, size = 2.4, colour = COL_TEXT, family = "sans"
  ) +
  scale_fill_gradient(low = COL_FLOOD_LOW, high = COL_FLOOD_HI, guide = "none") +
  scale_y_continuous(
    labels = scales::label_comma(scale = 1e-3, suffix = "k"),
    expand = expansion(mult = c(0, 0.2))
  ) +
  coord_flip() +
  labs(
    title    = "PEOPLE AT RISK",
    subtitle = "Population inside flood zones by sub-county",
    x = NULL, y = NULL
  ) +
  theme_minimal(base_family = "sans") +
  theme(
    plot.background  = element_rect(fill = COL_BG, colour = NA),
    panel.background = element_rect(fill = COL_BG, colour = NA),
    panel.grid.major.y = element_blank(),
    panel.grid.minor   = element_blank(),
    panel.grid.major.x = element_line(colour = "#D9D0BC", linewidth = 0.3),
    plot.title    = element_text(size = 9, face = "bold", colour = COL_TEXT, hjust = 0),
    plot.subtitle = element_text(size = 7, colour = COL_TEXT, hjust = 0),
    axis.text     = element_text(size = 6.5, colour = COL_TEXT),
    plot.margin   = margin(8, 16, 8, 8)
  )

# 8b. CHART — Roads at risk per subcounty

chart_roads <- ggplot(
  subcounty_stats %>% arrange(desc(roads_risk_km)) %>% slice_head(n = 10),
  aes(
    x    = reorder(NAME_2, roads_risk_km),
    y    = roads_risk_km,
    fill = roads_risk_km
  )
) +
  geom_col(width = 0.7) +
  geom_text(
    aes(label = paste0(roads_risk_km, " km")),
    hjust = -0.1, size = 2.4, colour = COL_TEXT, family = "sans"
  ) +
  scale_fill_gradient(low = COL_ROAD_RISK, high = "#B8860B", guide = "none") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.2))) +
  coord_flip() +
  labs(
    title    = "ROADS AT RISK",
    subtitle = "Flood-zone road length (km) by sub-county",
    x = NULL, y = NULL
  ) +
  theme_minimal(base_family = "sans") +
  theme(
    plot.background  = element_rect(fill = COL_BG, colour = NA),
    panel.background = element_rect(fill = COL_BG, colour = NA),
    panel.grid.major.y = element_blank(),
    panel.grid.minor   = element_blank(),
    panel.grid.major.x = element_line(colour = "#D9D0BC", linewidth = 0.3),
    plot.title    = element_text(size = 9, face = "bold", colour = COL_TEXT, hjust = 0),
    plot.subtitle = element_text(size = 7, colour = COL_TEXT, hjust = 0),
    axis.text     = element_text(size = 6.5, colour = COL_TEXT),
    plot.margin   = margin(8, 16, 8, 8)
  )

# 8c. CHART — Percentage of people affected per subcounty

chart_pct <- ggplot(
  subcounty_stats %>% arrange(desc(pct_at_risk)) %>% slice_head(n = 10),
  aes(
    x    = reorder(NAME_2, pct_at_risk),
    y    = pct_at_risk,
    fill = pct_at_risk
  )
) +
  geom_col(width = 0.7) +
  geom_text(
    aes(label = paste0(round(pct_at_risk, 1), "%")),
    hjust = -0.1, size = 2.4, colour = COL_TEXT, family = "sans"
  ) +
  scale_fill_gradient(low = alpha(COL_POP, 0.5), high = COL_POP, guide = "none") +
  scale_y_continuous(
    labels = scales::percent_format(scale = 1, accuracy = 1),
    expand = expansion(mult = c(0, 0.2))
  ) +
  coord_flip() +
  labs(
    title    = "% POPULATION AFFECTED",
    subtitle = "Share of sub-county residents inside flood zones",
    x = NULL, y = NULL
  ) +
  theme_minimal(base_family = "sans") +
  theme(
    plot.background  = element_rect(fill = COL_BG, colour = NA),
    panel.background = element_rect(fill = COL_BG, colour = NA),
    panel.grid.major.y = element_blank(),
    panel.grid.minor   = element_blank(),
    panel.grid.major.x = element_line(colour = "#D9D0BC", linewidth = 0.3),
    plot.title    = element_text(size = 9, face = "bold", colour = COL_TEXT, hjust = 0),
    plot.subtitle = element_text(size = 7, colour = COL_TEXT, hjust = 0),
    axis.text     = element_text(size = 6.5, colour = COL_TEXT),
    plot.margin   = margin(8, 16, 8, 8)
  )

# Preview all three
chart_people
chart_roads
chart_pct


## 4. Final Analysis and Visualisations

**Layer order (bottom to top):**

1. Population density \(warm amber raster\) \- context for exposure
2. KDE flood contour fills \- rings of recurrence intensity
3. KDE contour outlines \- white rings for visual separation
4. Safe roads \(grey\) \- infrastructure context
5. At\-risk roads \(amber dashed\) \- the analysis output
6. Neighbourhood labels \- geographic orientation

### Contour Threshold Tuning

The KDE density breaks are set at the **60th, 75th, 88th, 95th, and 99th percentiles** of non-zero density values. Adjust these thresholds to show more or fewer flood rings:

- Lower percentiles → more area classified as flood-prone
- Higher percentiles → only most chronic hotspots shown

### Chart Outputs

Three supporting bar charts show sub-county level exposure:

- **People at risk** — absolute population inside flood zones
- **Roads at risk** — total km of flood-affected road per sub-county  
- **% population affected** — relative exposure (highlights smaller but highly exposed sub-counties)

> **Export requirement:** Save at least 2 visualisations as PDF or PNG. Use `ggsave()` with `dpi = 300` for print\-quality output.



In [0]:
# 9. MAIN MAP

message("Building main map...")

# KDE density breaks — adjust after inspecting your data
# These quantile breaks define the contour ring thresholds
dens_breaks <- quantile(kde_df$density[kde_df$density > 0],
                        probs = c(0.60, 0.75, 0.88, 0.95, 0.99),
                        na.rm = TRUE)

main_map <- ggplot() +

  # --- Population density (warm amber glow beneath everything) ---
  geom_raster(
    data = pop_df,
    aes(x = lon, y = lat, fill = population),
    alpha = 0.55
  ) +
  scale_fill_gradient(
    low  = "transparent",
    high = COL_POP,
    na.value = "transparent",
    name = "Population/ndensity",
    guide = guide_colourbar(
      title.position = "top",
      barwidth = 0.5, barheight = 3,
      ticks = FALSE
    )
  ) +

  # --- Flood recurrence contour rings ---
  # (new scale so we use ggnewscale or manual approach)
  # We use stat_contour_filled on kde_df density
  new_scale_fill() +   # requires ggnewscale package
  stat_contour_filled(
    data    = kde_df,
    aes(x = lon, y = lat, z = density, fill = after_stat(level)),
    breaks  = dens_breaks,
    alpha   = 0.72
  ) +
  scale_fill_manual(
    values = c(
      alpha(COL_FLOOD_LOW, 0.5),
      alpha(COL_FLOOD_MID, 0.55),
      alpha(COL_FLOOD_MID, 0.75),
      alpha(COL_FLOOD_HI,  0.85),
      alpha(COL_FLOOD_HI,  0.95)
    ),
    name   = "Flood recurrence/n(2014–2024)",
    labels = c("Occasional", "Moderate", "High", "Severe", "Chronic"),
    guide  = guide_legend(
      title.position = "top",
      override.aes   = list(alpha = 1)
    )
  ) +

  # --- Contour ring outlines (the "rings of risk" effect) ---
  stat_contour(
    data      = kde_df,
    aes(x = lon, y = lat, z = density),
    breaks    = dens_breaks,
    colour    = "white",
    linewidth = 0.25,
    alpha     = 0.6,
    linetype  = "solid"
  ) +

  # --- Roads — safe (outside flood zone) ---
  geom_sf(
    data      = roads_at_risk %>% filter(!in_flood_zone, road_class == "Major"),
    colour    = COL_ROAD,
    linewidth = 0.35,
    alpha     = 0.9
  ) +
  geom_sf(
    data      = roads_at_risk %>% filter(!in_flood_zone, road_class != "Major"),
    colour    = COL_ROAD,
    linewidth = 0.12,
    alpha     = 0.6
  ) +

  # --- Roads — AT RISK (inside flood zone) — rendered as dashed amber ---
  geom_sf(
    data      = roads_at_risk %>% filter(in_flood_zone, road_class == "Major"),
    colour    = COL_ROAD_RISK,
    linewidth = 0.5,
    linetype  = "dashed",
    alpha     = 1
  ) +
  geom_sf(
    data      = roads_at_risk %>% filter(in_flood_zone, road_class != "Major"),
    colour    = COL_ROAD_RISK,
    linewidth = 0.2,
    linetype  = "dashed",
    alpha     = 0.85
  ) +

  # --- Neighbourhood labels ---
  geom_label_repel(
    data         = nairobi_labels,
    aes(x = lon, y = lat, label = name),
    size         = 2.4,
    family       = "sans",
    colour       = COL_TEXT,
    fill         = alpha(COL_BG, 0.85),
    label.size   = 0,
    label.padding = unit(0.15, "lines"),
    segment.colour = COL_TEXT,
    segment.size = 0.3,
    min.segment.length = 0.2,
    box.padding  = 0.4,
    force        = 2,
    seed         = 42
  ) +

  # --- Map extent & coords ---
  coord_sf(
    xlim = c(nairobi_bbox["xmin"], nairobi_bbox["xmax"]),
    ylim = c(nairobi_bbox["ymin"], nairobi_bbox["ymax"]),
    expand = FALSE
  ) +

  # --- Scale bar & north arrow ---
  annotation_scale(
    location   = "bl",
    width_hint = 0.18,
    text_col   = COL_TEXT,
    line_col   = COL_TEXT,
    bar_cols   = c(COL_TEXT, COL_BG)
  ) +
  annotation_north_arrow(
    location = "bl",
    pad_x = unit(0.5, "cm"), pad_y = unit(0.7, "cm"),
    style = north_arrow_minimal(fill = COL_TEXT, line_col = COL_TEXT,
                                text_col = COL_TEXT, text_size = 7)
  ) +

  # --- Theme ---
  theme_void(base_family = "sans") +
  theme(
    plot.background  = element_rect(fill = COL_BG, colour = NA),
    panel.background = element_rect(fill = "#D6CCBA", colour = NA),
    legend.position  = c(0.02, 0.98),
    legend.justification = c(0, 1),
    legend.background = element_rect(fill = alpha(COL_BG, 0.85), colour = NA),
    legend.key.size   = unit(0.35, "cm"),
    legend.title      = element_text(size = 7,  face = "bold", colour = COL_TEXT),
    legend.text       = element_text(size = 6.5, colour = COL_TEXT),
    legend.margin     = margin(4, 6, 4, 6),
    legend.box.margin = margin(0, 0, 0, 0),
    plot.margin       = margin(0, 0, 0, 0)
  )

# 10. CALLOUT STAT BOXES

make_stat_box <- function(number, label, bg = COL_FLOOD_MID) {
  ggplot() +
    annotate("rect",
             xmin = 0, xmax = 1, ymin = 0, ymax = 1,
             fill = bg, colour = NA) +
    annotate("text",
             x = 0.5, y = 0.62, label = number,
             size = 7, fontface = "bold",
             colour = "white", family = "sans", hjust = 0.5) +
    annotate("text",
             x = 0.5, y = 0.25, label = label,
             size = 2.5, colour = "white",
             family = "sans", hjust = 0.5, lineheight = 1.1) +
    theme_void() +
    theme(plot.background = element_rect(fill = bg, colour = NA))
}

stat1 <- make_stat_box(
  scales::comma(round(pop_in_chronic, -3)),
  "people live inside/nchronic flood zones",
  bg = COL_FLOOD_HI
)

stat2 <- make_stat_box(
  as.character(roads_risk_km),
  "km of roads regularly/nunder flood threat",
  bg = COL_FLOOD_MID
)

stat3 <- make_stat_box(
  "10 yrs",
  "of Sentinel-1 SAR/nflood observations",
  bg = "#5C4033"
)

stats_row <- plot_grid(stat1, stat2, stat3, nrow = 1, rel_widths = c(1, 1, 1))

# 11. TITLE & SOURCE PANEL

title_panel <- ggplot() +
  theme_void() +
  theme(plot.background = element_rect(fill = COL_BG, colour = NA)) +
  annotate("text",
           x = 0.03, y = 0.80,
           label = "RINGS OF RISK",
           size = 11, fontface = "bold",
           colour = COL_TEXT, family = "sans", hjust = 0) +
  annotate("text",
           x = 0.03, y = 0.52,
           label = "Nairobi's flood hotspots and the walking routes they sever",
           size = 5, colour = COL_TEXT, family = "sans", hjust = 0) +
  annotate("text",
           x = 0.03, y = 0.28,
           label = paste0(
             "Contour rings show where flooding has been detected repeatedly over 10 years (2014–2024) ",
             "using cloud-penetrating/nSentinel-1 SAR satellite data. Amber dashed lines are road segments ",
             "falling inside high or chronic flood zones —/nroutes on which pedestrians and cyclists depend ",
             "daily, with no dedicated infrastructure to protect them."
           ),
           size = 2.8, colour = COL_TEXT, family = "sans", hjust = 0,
           lineheight = 1.35) +
  annotate("text",
           x = 0.03, y = 0.05,
           label = paste0(
             "Sources: AI for Good Lab / Microsoft Flood Dataset (Nature Communications, 2025)  •  ",
             "OpenStreetMap  •  WorldPop Kenya 2020/n",
             "Methodology: Flood detections filtered for false positives (slope, soil moisture, temperature, land cover). ",
             "KDE weighted by recurrence months. All roads assumed to carry pedestrian & cyclist traffic."
           ),
           size = 2.2, colour = alpha(COL_TEXT, 0.65),
           family = "sans", hjust = 0, lineheight = 1.3) +
  xlim(0, 1) + ylim(0, 1)

# 12. LEGEND STRIP — road types

road_legend <- ggplot() +
  theme_void() +
  theme(plot.background = element_rect(fill = COL_BG, colour = NA)) +
  annotate("segment",
           x = 0.05, xend = 0.22, y = 0.65, yend = 0.65,
           colour = COL_ROAD, linewidth = 0.8) +
  annotate("text",
           x = 0.25, y = 0.65, label = "Road — no flood risk",
           size = 2.6, colour = COL_TEXT, hjust = 0) +
  annotate("segment",
           x = 0.05, xend = 0.22, y = 0.35, yend = 0.35,
           colour = COL_ROAD_RISK, linewidth = 0.8, linetype = "dashed") +
  annotate("text",
           x = 0.25, y = 0.35, label = "Road — inside flood zone",
           size = 2.6, colour = COL_TEXT, hjust = 0) +
  xlim(0, 1) + ylim(0, 1)

# 13. ASSEMBLE FULL POSTER

message("Assembling poster...")

# Require ggnewscale for dual fill scales in main map
# install.packages("ggnewscale") if not present

top_panel    <- plot_grid(title_panel, road_legend,
                          nrow = 1, rel_widths = c(3, 1))
charts_row <- plot_grid(chart_people, chart_roads,
                        nrow = 1, rel_widths = c(1, 1))
bottom_panel <- plot_grid(stats_row, charts_row,
                          nrow = 1, rel_widths = c(0.6, 1.4))

full_poster <- plot_grid(
  top_panel,
  main_map,
  bottom_panel,
  chart_pct,
  ncol        = 1,
  rel_heights = c(0.15, 0.55, 0.18, 0.12)
)

# 14. SAVE

output_path <- "outputs/rings_of_risk_nairobi.png"
message(glue::glue("Saving to {output_path}..."))

ggsave(
  filename = output_path,
  plot     = full_poster,
  width    = 14,
  height   = 17,
  dpi      = 300,
  bg       = COL_BG
)

message("Done! Open rings_of_risk_nairobi.png")

# 15. EXPORT VARIATIONS

message("Saving poster variations...")

# --- Variation A: Light / Print-friendly (default parchment palette) ---
ggsave(
  filename = "outputs/rings_of_risk_nairobi_LIGHT.pdf",
  plot     = full_poster,
  width    = 14, height = 17,
  bg       = COL_BG
)

# --- Variation B: Dark / Presentation mode ---
COL_BG_DARK      <- "#1A1410"
COL_TEXT_DARK    <- "#F0EAD6"
COL_ROAD_DARK    <- "#4A4438"
COL_ROAD_RISK_DK <- "#FFD700"

full_poster_dark <- full_poster &
  theme(
    plot.background  = element_rect(fill = COL_BG_DARK, colour = NA),
    panel.background = element_rect(fill = "#0F0C08",    colour = NA),
    text             = element_text(colour = COL_TEXT_DARK)
  )

ggsave(
  filename = "outputs/rings_of_risk_nairobi_DARK.png",
  plot     = full_poster_dark,
  width    = 14, height = 17, dpi = 300,
  bg       = COL_BG_DARK
)

# --- Variation C: High-contrast / Accessibility (larger text, bolder fills) ---
full_poster_hc <- full_poster &
  theme(
    text      = element_text(size = 10, colour = COL_TEXT),
    axis.text = element_text(size = 8),
    legend.text  = element_text(size = 8),
    legend.title = element_text(size = 9, face = "bold")
  )

ggsave(
  filename = "outputs/rings_of_risk_nairobi_HIGHCONTRAST.png",
  plot     = full_poster_hc,
  width    = 14, height = 17, dpi = 300,
  bg       = COL_BG
)

message("All variations saved to outputs/")



# APPENDIX: PACKAGE INSTALL BLOCK (run once before the script above)

# install.packages(c(
#   "tidyverse", "sf", "terra", "osmdata", "MASS",
#   "scales", "patchwork", "ggspatial", "ggrepel",
#   "cowplot", "arrow", "ggnewscale", "glue", "units"
# ))
#
# WorldPop download:
#   https://hub.worldpop.org/geodata/listing?id=29786
#   → Kenya 2020, unconstrained, 100m: KEN_ppp_v2b_2020_UNadj.tif
#
# Flood tile:
#   https://huggingface.co/datasets/ai-for-good-lab/ai4g-flood-dataset/tree/main
#   → Folder: S03/S03E036/
#   → Files needed:
#       S03E036-post-processing.parquet
#       S03E036-recurrence-80m-buffer.tif


In [0]:
# =========================
# 9. MAIN MAP (robust)
# =========================
library(ggplot2)
library(dplyr)
library(ggnewscale)
library(ggrepel)
library(ggspatial)
library(sf)

message("Building main map...")

# ---- 9.0 Ensure plotting layers (roads) are in WGS84 for the map ----
if (!exists("roads_at_risk_wgs")) {
  stopifnot(inherits(roads_at_risk, "sf"))
  roads_at_risk_wgs <- sf::st_transform(roads_at_risk, 4326)
}
if (exists("roads_flood_segments") && !exists("roads_flood_segments_wgs")) {
  roads_flood_segments_wgs <- sf::st_transform(roads_flood_segments, 4326)
}

# ---- 9.1 KDE density breaks with a safe fallback ----
dens_vec <- kde_df$density
dens_vec <- dens_vec[is.finite(dens_vec) & dens_vec > 0]

if (length(dens_vec) >= 10) {
  dens_breaks <- stats::quantile(
    dens_vec, probs = c(0.60, 0.75, 0.88, 0.95, 0.99), na.rm = TRUE
  )
} else if (length(dens_vec) > 0) {
  dens_breaks <- stats::quantile(
    dens_vec, probs = c(0.50, 0.70, 0.85, 0.95), na.rm = TRUE
  )
} else {
  dens_breaks <- c(1e-9, 2e-9, 3e-9, 4e-9)  # renders “nothing” but avoids errors
  warning("KDE density appears empty; contour rings may not render.")
}

# ---- 9.2 Prefer clipped flooded segments if available ----
use_segments <- exists("roads_flood_segments_wgs") &&
                inherits(roads_flood_segments_wgs, "sf") &&
                nrow(roads_flood_segments_wgs) > 0

# ---- 9.3 Build the map ----
main_map <- ggplot() +

  # --- Population density (warm amber glow beneath everything) ---
  geom_raster(
    data = pop_df,
    aes(x = lon, y = lat, fill = population),
    alpha = 0.65
  ) +
  scale_fill_gradient(
    low  = "transparent",
    high = COL_POP,
    na.value = "transparent",
    name = "Population\ndensity",    # <-- fixed line break
    guide = guide_colourbar(
      title.position = "top",
      barwidth = 0.5, barheight = 3,
      ticks = FALSE
    )
  ) +

  # --- Flood recurrence contour rings (KDE) ---
  ggnewscale::new_scale_fill() +
  stat_contour_filled(
    data    = kde_df,
    aes(x = lon, y = lat, z = density, fill = after_stat(level)),
    breaks  = dens_breaks,
    alpha   = 0.72
  ) +
  scale_fill_manual(
    values = c(
      alpha(COL_FLOOD_LOW, 0.5),
      alpha(COL_FLOOD_MID, 0.55),
      alpha(COL_FLOOD_MID, 0.75),
      alpha(COL_FLOOD_HI,  0.85),
      alpha(COL_FLOOD_HI,  0.95)
    ),
    name   = "Flood recurrence\n(2014–2024)",  # <-- fixed line break
    labels = c("Occasional", "Moderate", "High", "Severe", "Chronic"),
    guide  = guide_legend(
      title.position = "top",
      override.aes   = list(alpha = 1)
    )
  ) +

  # --- Contour ring outlines (the "rings of risk" effect) ---
  stat_contour(
    data      = kde_df,
    aes(x = lon, y = lat, z = density),
    breaks    = dens_breaks,
    colour    = "white",
    linewidth = 0.25,
    alpha     = 0.6,
    linetype  = "solid"
  ) +

  # --- Roads — safe (outside flood zone) ---
  geom_sf(
    data      = subcounties,
    fill      = alpha(COL_FLOOD_LOW, 0.08),
    colour    = alpha(COL_FLOOD_MID, 0.55),
    linewidth = 0.4,
    linetype  = "solid"
  ) + geom_sf(
    data      = roads_at_risk_wgs %>% dplyr::filter(!in_flood_zone, road_class == "Major"),
    colour    = COL_ROAD,
    linewidth = 0.35,
    alpha     = 0.9
  ) +
  geom_sf(
    data      = roads_at_risk_wgs %>% dplyr::filter(!in_flood_zone, road_class != "Major"),
    colour    = COL_ROAD,
    linewidth = 0.12,
    alpha     = 0.6
  ) +

  # --- Roads — AT RISK (inside flood zone), dashed amber ---
  {
    if (use_segments) {
      list(
        geom_sf(
          data      = roads_flood_segments_wgs %>% dplyr::filter(road_class == "Major"),
          colour    = COL_ROAD_RISK,
          linewidth = 0.5,
          linetype  = "dashed",
          alpha     = 1
        ),
        geom_sf(
          data      = roads_flood_segments_wgs %>% dplyr::filter(road_class != "Major"),
          colour    = COL_ROAD_RISK,
          linewidth = 0.2,
          linetype  = "dashed",
          alpha     = 0.85
        )
      )
    } else {
      list(
        geom_sf(
          data      = roads_at_risk_wgs %>% dplyr::filter(in_flood_zone, road_class == "Major"),
          colour    = COL_ROAD_RISK,
          linewidth = 0.5,
          linetype  = "dashed",
          alpha     = 1
        ),
        geom_sf(
          data      = roads_at_risk_wgs %>% dplyr::filter(in_flood_zone, road_class != "Major"),
          colour    = COL_ROAD_RISK,
          linewidth = 0.2,
          linetype  = "dashed",
          alpha     = 0.85
        )
      )
    }
  } +

  # --- Neighbourhood labels ---
  ggrepel::geom_label_repel(
    data           = nairobi_labels,     # tibble with lon/lat/name in EPSG:4326
    aes(x = lon, y = lat, label = name),
    size           = 2.4,
    family         = "sans",
    colour         = COL_TEXT,
    fill           = alpha(COL_BG, 0.85),
    label.size     = 0,
    label.padding  = unit(0.15, "lines"),
    segment.colour = COL_TEXT,
    segment.size   = 0.3,
    min.segment.length = 0.2,
    box.padding    = 0.4,
    force          = 2,
    seed           = 42
  ) +

  # --- Map extent & coords (WGS84) ---
  coord_sf(
    xlim = c(nairobi_bbox["xmin"], nairobi_bbox["xmax"]),
    ylim = c(nairobi_bbox["ymin"], nairobi_bbox["ymax"]),
    expand = FALSE
  ) +

  # --- Scale bar & north arrow ---
  ggspatial::annotation_scale(
    location   = "bl",
    width_hint = 0.18,
    text_col   = COL_TEXT,
    line_col   = COL_TEXT,
    bar_cols   = c(COL_TEXT, COL_BG)
  ) +
  ggspatial::annotation_north_arrow(
    location = "bl",
    pad_x = unit(0.5, "cm"), pad_y = unit(0.7, "cm"),
    style = ggspatial::north_arrow_minimal(
      fill = COL_TEXT, line_col = COL_TEXT, text_col = COL_TEXT, text_size = 7
    )
  ) +

  # --- Theme ---
  theme_void(base_family = "sans") +
  theme(
    plot.background  = element_rect(fill = COL_BG, colour = NA),
    panel.background = element_rect(fill = "#D6CCBA", colour = NA),
    legend.position  = c(0.02, 0.98),
    legend.justification = c(0, 1),
    legend.background = element_rect(fill = alpha(COL_BG, 0.85), colour = NA),
    legend.key.size   = unit(0.35, "cm"),
    legend.title      = element_text(size = 7,  face = "bold", colour = COL_TEXT),
    legend.text       = element_text(size = 6.5, colour = COL_TEXT),
    legend.margin     = margin(4, 6, 4, 6),
    legend.box.margin = margin(0, 0, 0, 0),
    plot.margin       = margin(0, 0, 0, 0)
  )
main_map

## 5. Ethical Considerations

### Privacy

The dataset contains no personal data. All flood detections are spatially aggregated from satellite imagery. Population figures are modelled estimates (WorldPop/Meta HRSL), not individual records.

### Data Provenance & Licensing

| Dataset | License | Citation Required |
| :------------------- | :-------- | :------------------------------------------------------------------------------ |
| AI4G Flood Dataset | MIT | Yes \- [Nature Communications 2025](https://doi.org/10.1038/s41467-025-60973-1) |
| WorldPop / Meta HRSL | CC BY 4.0 | Yes |
| OpenStreetMap | ODbL | Yes \- © OpenStreetMap contributors |

### Potential Misuse

Flood risk maps can influence property values and insurance assessments. Care should be taken that:

- Results are communicated with appropriate uncertainty bounds
- Communities are not stigmatised or displaced based on modelled risk alone
- Decisions affecting residents incorporate ground-truth validation

### Model Limitations

- SAR-based detection can produce **false negatives in urban areas** (buildings cause radar shadow effects)
- Detection quality is **reduced under dense forest canopy**
- The 10-year record may not capture long-term climate trend changes in flood frequency
- Filters are tuned for global applicability; local calibration improves accuracy

### Positionality

This analysis was conducted for Nairobi, Kenya \- a city where the researcher has direct local knowledge. Interpretations of neighbourhood names and flood impact should be validated with community members and local government data where possible.
